In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# Load data
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')


# Save test IDs
test_ids = test['Id']

# Target (log transform)
y = np.log1p(train['SalePrice'])

# Drop target and IDs
train.drop(['SalePrice', 'Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

# Combine for preprocessing
combined = pd.concat([train, test], axis=0)

# Fill missing values
for col in combined.columns:
    if combined[col].dtype == 'object':
        combined[col] = combined[col].fillna('Missing')
    else:
        combined[col] = combined[col].fillna(combined[col].median())

# One-hot encode categorical variables
combined = pd.get_dummies(combined)

# Split back
X = combined.iloc[:len(train), :]
test_data = combined.iloc[len(train):, :]

# Train-validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# XGBoost model
model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=1,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Validate
y_pred = model.predict(X_valid)

# Metrics
r2 = r2_score(y_valid, y_pred)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
mae = mean_absolute_error(y_valid, y_pred)

print("R2 Score:", r2)
print("RMSE:", rmse)
print("MAE:", mae)

#Train full model
model.fit(X, y)

# Predict test set
predictions = model.predict(test_data)

# Reverse log transform
predictions = np.expm1(predictions)

# Submission
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv('submission2.csv', index=False)

print("submission.csv created successfully")
print(submission.head())

R2 Score: 0.9087778560757397
RMSE: 0.13047242796436764
MAE: 0.08478378299279503
submission.csv created successfully
     Id      SalePrice
0  1461  124001.132812
1  1462  158991.156250
2  1463  191728.953125
3  1464  194946.343750
4  1465  183945.343750
